# R&D-7 — внутренний контур: два скрипта обработки сырых эмбеддингов

Самодостаточный ноутбук (pyspark, без импортов `rnd_reports`). Сырые данные — таблица формата
`epk_id × report_dt × emb_{i}_val` (i = 0..n). Внутри — **два скрипта**:

1. **Скрипт 1** — `EmbeddingReducer`: понижение размерности эмбеддингов до `red_size`
   (StandardScaler + PCA) → `[epk_id, report_dt, red_0, …, red_{red_size-1}]`.
2. **Скрипт 2** — `PropensityScorer`: берёт ту же таблицу эмбеддингов + таблицу псевдо-разбиения
   `epk_id × treatment` и выдаёт `prop_score` = P(treatment=1), обучив лучший из
   **LogisticRegression / GBTClassifier** (по ROC-AUC на отложенной выборке).

`prop_score` дальше пойдёт в propensity-score matching по Рубину — **сам матчинг здесь не реализуется**.

**Как использовать во внутреннем контуре:**
- в ячейке конфигурации укажите свои spark-таблицы (`spark.table(...)` / свои пути) в
  `INPUT_EMB` и `INPUT_TREATMENT` и нужный `RED_SIZE`;
- **пропустите** ячейку-генератор синтетики (помечена «НЕ ЗАПУСКАТЬ ВО ВНУТРЕННЕМ КОНТУРЕ»);
- выполните остальные ячейки по порядку.


In [1]:
# Импорты — только базовый pyspark-стек (без rnd_reports).
import re

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.ml.feature import PCA, StandardScaler, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.classification import GBTClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

## Библиотека (вшита из `rnd_reports.embeddings`)

Контракт схемы + оба скрипта-адаптера. Во внутреннем контуре эти ячейки не правят.

In [2]:
# === Вшито из rnd_reports.embeddings/contracts.py + reducer.py ===
# Контракт сырой таблицы: epk_id, report_dt, emb_0_val, emb_1_val, ... (легаси-формат col_000 тоже понимается).

EPK_ID, REPORT_DT, TREATMENT = "epk_id", "report_dt", "treatment"
KEY_COLUMNS = (EPK_ID, REPORT_DT)
REDUCED_PREFIX = "red_"          # выход reducer'а: red_0, red_1, ...
PROPENSITY_SCORE = "prop_score"  # выход propensity-адаптера

_EMB_NEW = re.compile(r"^emb_(\d+)_val$")
_EMB_LEGACY = re.compile(r"^col_(\d+)$")


def _embedding_index(name):
    m = _EMB_NEW.match(name) or _EMB_LEGACY.match(name)
    return int(m.group(1)) if m else None


def embedding_feature_columns(df):
    indexed = [(i, c) for c in df.columns if (i := _embedding_index(c)) is not None]
    return [name for _, name in sorted(indexed)]


def reduced_column_names(red_size):
    if red_size < 1:
        raise ValueError(f"red_size должен быть >= 1, получено {red_size}")
    return [f"{REDUCED_PREFIX}{i}" for i in range(red_size)]


def validate_embedding_schema(df):
    missing = [c for c in KEY_COLUMNS if c not in set(df.columns)]
    if missing:
        raise ValueError(f"В эмбеддинг-датасете нет ключевых колонок: {missing}")
    cols = embedding_feature_columns(df)
    if not cols:
        raise ValueError("Не найдено ни одной эмбеддинг-колонки 'emb_{i}_val' (или легаси 'col_{i}')")
    return cols


def validate_treatment_schema(df):
    missing = [c for c in (EPK_ID, TREATMENT) if c not in set(df.columns)]
    if missing:
        raise ValueError(f"В датасете трита нет колонок: {missing}")


def treatment_join_keys(df):
    # Джойн по (epk_id, report_dt), если в треате есть report_dt, иначе по epk_id.
    return list(KEY_COLUMNS) if REPORT_DT in set(df.columns) else [EPK_ID]


class EmbeddingReducer:
    """Скрипт 1: понижение размерности эмбеддингов до red_size (StandardScaler + PCA).

    fit(df, cutoff) -> transform(df); при cutoff обучение только на report_dt <= cutoff (in-time safety).
    """

    def __init__(self, red_size=5, seed=42):
        if red_size < 1:
            raise ValueError(f"red_size должен быть >= 1, получено {red_size}")
        self.red_size, self.seed = red_size, seed
        self._assembler = self._scaler_model = self._pca_model = None

    @property
    def is_fitted(self):
        return self._pca_model is not None

    def fit(self, df, cutoff=None):
        feats = validate_embedding_schema(df)
        train = df if cutoff is None else df.filter(F.col(REPORT_DT) <= F.lit(cutoff))
        self._assembler = VectorAssembler(inputCols=feats, outputCol="__f__")
        assembled = self._assembler.transform(train)
        self._scaler_model = StandardScaler(
            inputCol="__f__", outputCol="__s__", withMean=True, withStd=True
        ).fit(assembled)
        scaled = self._scaler_model.transform(assembled)
        self._pca_model = PCA(k=self.red_size, inputCol="__s__", outputCol="__p__").fit(scaled)
        return self

    def transform(self, df):
        if not self.is_fitted:
            raise RuntimeError("EmbeddingReducer не обучен: вызовите fit(...) до transform(...)")
        validate_embedding_schema(df)
        out = self._pca_model.transform(self._scaler_model.transform(self._assembler.transform(df)))
        out = out.withColumn("__a__", vector_to_array(F.col("__p__")))
        names = reduced_column_names(self.red_size)
        cols = [F.col(EPK_ID), F.col(REPORT_DT)] + [F.col("__a__")[i].alias(n) for i, n in enumerate(names)]
        return out.select(*cols)

    def fit_transform(self, df, cutoff=None):
        return self.fit(df, cutoff=cutoff).transform(df)

In [3]:
# === Вшито из rnd_reports.embeddings/propensity.py ===
# Скрипт 2: prop_score = P(treatment=1 | эмбеддинги); лучший из LogReg / GBT по ROC-AUC.


class PropensityScorer:
    """fit(embeddings_df, treatment_df, cutoff) -> transform(embeddings_df) с колонкой prop_score.

    После fit доступны metrics_ (AUC обеих моделей) и best_model_name_.
    """

    def __init__(self, max_iter=100, reg_param=0.0, gbt_max_depth=5, gbt_max_iter=20,
                 val_fraction=0.3, seed=42):
        self.max_iter, self.reg_param = max_iter, reg_param
        self.gbt_max_depth, self.gbt_max_iter = gbt_max_depth, gbt_max_iter
        self.val_fraction, self.seed = val_fraction, seed
        self._assembler = self._model = None
        self.metrics_, self.best_model_name_ = {}, None

    @property
    def is_fitted(self):
        return self._model is not None

    def _candidates(self):
        # probability/raw задаём сеттерами: GBTClassifier не принимает их kwargs конструктора.
        est = {
            "logreg": LogisticRegression(featuresCol="__f__", labelCol="__y__",
                                         maxIter=self.max_iter, regParam=self.reg_param),
            "gbt": GBTClassifier(featuresCol="__f__", labelCol="__y__",
                                 maxDepth=self.gbt_max_depth, maxIter=self.gbt_max_iter, seed=self.seed),
        }
        for e in est.values():
            e.setProbabilityCol("__prob__").setRawPredictionCol("__raw__")
        return est

    def fit(self, embeddings_df, treatment_df, cutoff=None):
        feats = validate_embedding_schema(embeddings_df)
        validate_treatment_schema(treatment_df)
        keys = treatment_join_keys(treatment_df)
        joined = embeddings_df.join(treatment_df.select(*keys, TREATMENT), keys, "inner")
        if cutoff is not None:
            joined = joined.filter(F.col(REPORT_DT) <= F.lit(cutoff))
        labelled = joined.withColumn("__y__", F.col(TREATMENT).cast("double"))
        self._assembler = VectorAssembler(inputCols=feats, outputCol="__f__")
        assembled = self._assembler.transform(labelled)
        train, val = assembled.randomSplit([1 - self.val_fraction, self.val_fraction], seed=self.seed)
        ev = BinaryClassificationEvaluator(labelCol="__y__", rawPredictionCol="__raw__",
                                           metricName="areaUnderROC")
        for name, est in self._candidates().items():
            self.metrics_[name] = float(ev.evaluate(est.fit(train).transform(val)))
        self.best_model_name_ = max(self.metrics_, key=self.metrics_.get)
        self._model = self._candidates()[self.best_model_name_].fit(assembled)
        return self

    def transform(self, embeddings_df):
        if not self.is_fitted:
            raise RuntimeError("PropensityScorer не обучен: вызовите fit(...) до transform(...)")
        validate_embedding_schema(embeddings_df)
        scored = self._model.transform(self._assembler.transform(embeddings_df))
        scored = scored.withColumn("__pa__", vector_to_array(F.col("__prob__")))
        return scored.select(F.col(EPK_ID), F.col(REPORT_DT),
                             F.col("__pa__")[1].alias(PROPENSITY_SCORE))

## Конфигурация / контракт таблиц

`INPUT_EMB` — сырые эмбеддинги (`epk_id, report_dt, emb_0_val, …`).
`INPUT_TREATMENT` — псевдо-разбиение (`epk_id, treatment`).
Во внутреннем контуре подставьте свои таблицы (например `spark.table("schema.table")`).

In [4]:
# --- Конфигурация прогона ----------------------------------------------------------
import os, glob, shutil

# Bootstrap JAVA_HOME для локального Spark (во внутреннем контуре обычно уже задан).
if not (os.environ.get("JAVA_HOME") and os.path.isdir(os.environ["JAVA_HOME"])):
    cand = sorted(glob.glob("/usr/lib/jvm/*"), reverse=True)
    if (java := shutil.which("java")):
        home = os.path.realpath(java)
        for _ in range(3):  # .../<jdk>/jre/bin/java -> .../<jdk>
            home = os.path.dirname(home)
        cand.insert(0, home)
    for c in cand:
        if os.path.isdir(os.path.join(c, "bin")):
            os.environ["JAVA_HOME"] = c
            break

INPUT_EMB = "data/07_embedding_adjustment_set/emb_raw.parquet"          # во внутреннем контуре — своя таблица
INPUT_TREATMENT = "data/07_embedding_adjustment_set/treatment.parquet"  # epk_id × treatment
RED_SIZE = 5
SEED = 42

spark = (
    SparkSession.builder.master("local[1]").appName("rnd7-internal")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| JAVA_HOME =", os.environ.get("JAVA_HOME"))

Spark 3.4.4 | JAVA_HOME = /usr/lib/jvm/java-8-openjdk-amd64


In [5]:
# +++ ЯЧЕЙКА-ГЕНЕРАТОР СИНТЕТИКИ +++
# +++ НЕ ЗАПУСКАТЬ ВО ВНУТРЕННЕМ КОНТУРЕ +++
# Синтез ОДНОЙ демонстрационной пары таблиц (эмбеддинги + трит) и запись в parquet (INPUT_*).
# Нужна только для проверки запускаемости ноутбука в репозитории. Во внутреннем контуре —
# реальные таблицы, эту ячейку пропускают.
import numpy as np
import pandas as pd
from pathlib import Path

_rng = np.random.default_rng(SEED)
_n, _k, _n_latent = 4000, 16, 3
_z = _rng.normal(size=(_n, _n_latent))                       # латентные конфаундеры
_W = _rng.normal(size=(_n_latent, _k))
_emb = _z @ _W + _rng.normal(scale=0.5, size=(_n, _k))       # эмбеддинги = латенты + шум
_logit = _z @ _rng.normal(size=_n_latent) * 1.3             # трит по селекции на латентах -> AUC > 0.5
_treat = (_rng.uniform(size=_n) < 1 / (1 + np.exp(-_logit))).astype(int)
_months = np.array(["2024-01-01", "2024-02-01", "2024-03-01"])
_report_dt = _months[_rng.integers(0, len(_months), size=_n)]
_epk = 1000 + np.arange(_n)

_emb_pd = pd.DataFrame(_emb, columns=[f"emb_{j}_val" for j in range(_k)])
_emb_pd.insert(0, "report_dt", _report_dt)
_emb_pd.insert(0, "epk_id", _epk)
_trt_pd = pd.DataFrame({"epk_id": _epk, "treatment": _treat})

Path(INPUT_EMB).parent.mkdir(parents=True, exist_ok=True)
spark.createDataFrame(_emb_pd).write.mode("overwrite").parquet(INPUT_EMB)
spark.createDataFrame(_trt_pd).write.mode("overwrite").parquet(INPUT_TREATMENT)
print(f"синтетика записана: {_n} юнитов, {_k} эмбеддинг-колонок; доля treatment=1 = {_treat.mean():.3f}")

синтетика записана: 4000 юнитов, 16 эмбеддинг-колонок; доля treatment=1 = 0.498


In [6]:
# Загрузка таблиц. Во внутреннем контуре сюда приходят реальные таблицы из INPUT_*.
emb = spark.read.parquet(INPUT_EMB)
trt = spark.read.parquet(INPUT_TREATMENT)
print("эмбеддинги:", emb.count(), "строк,", len(embedding_feature_columns(emb)), "колонок emb_*_val")
print("трит:", trt.count(), "строк; доля treatment=1 =",
      round(trt.selectExpr("avg(treatment)").first()[0], 3))
emb.limit(5).toPandas()

эмбеддинги: 4000 строк, 16 колонок emb_*_val
трит: 4000 строк; доля treatment=1 = 0.498


,epk_id,report_dt,emb_0_val,emb_1_val,emb_2_val,emb_3_val,emb_4_val,emb_5_val,emb_6_val,emb_7_val,emb_8_val,emb_9_val,emb_10_val,emb_11_val,emb_12_val,emb_13_val,emb_14_val,emb_15_val
0,1000,2024-01-01,0.386700,2.461619,-2.524029,1.990745,0.964076,1.547253,-4.229097,-0.591109,-0.626682,-0.206244,0.380927,-2.704960,0.684516,-0.877203,-0.824659,-0.477414
1,1001,2024-03-01,-2.105155,3.231162,0.878168,-1.462853,-0.306624,-1.225297,-0.511273,-2.585024,-1.046876,-1.114414,0.689469,-4.569152,4.261411,-1.299537,-3.038901,-1.419793
2,1002,2024-01-01,-0.347417,0.233247,-0.315198,-0.475349,-0.279976,-0.228226,-0.021323,-0.412320,-0.850183,-0.936599,0.213891,-0.824945,0.099039,-0.405309,-0.233105,0.432838
3,1003,2024-02-01,0.574019,-1.055511,-0.130157,1.044985,-0.115584,1.579408,-0.344418,0.398490,0.520068,0.498321,-0.679206,2.520896,-1.024178,2.795302,2.262314,0.565890
4,1004,2024-02-01,0.844067,-1.939272,-0.974482,-0.999793,-0.675547,0.338302,0.152804,1.530257,0.413706,1.421441,-0.042543,3.637684,-2.020274,-0.755419,1.728385,1.142775


## Скрипт 1 — понижение размерности эмбеддингов

`EmbeddingReducer(red_size=RED_SIZE)`: StandardScaler + PCA → `[epk_id, report_dt, red_0, …]`.
Опционально `cutoff=...` для in-time safety (обучение только на `report_dt <= cutoff`).

In [7]:
reducer = EmbeddingReducer(red_size=RED_SIZE)
reduced = reducer.fit_transform(emb)      # во внутреннем контуре можно fit_transform(emb, cutoff="2024-02-01")
print("выход reducer:", reduced.columns)
reduced.limit(5).toPandas()

выход reducer: ['epk_id', 'report_dt', 'red_0', 'red_1', 'red_2', 'red_3', 'red_4']


,epk_id,report_dt,red_0,red_1,red_2,red_3,red_4
0,1000,2024-01-01,0.622190,-2.362363,-0.023584,-1.975121,1.058780
1,1001,2024-03-01,3.855902,-1.364080,-3.186711,0.191165,-0.406139
2,1002,2024-01-01,0.408155,-0.526132,-0.184053,0.297947,-0.206375
3,1003,2024-02-01,-1.111463,0.797809,2.164453,0.284795,0.700500
4,1004,2024-02-01,-2.564040,0.342267,1.439965,1.159731,-0.785371


## Скрипт 2 — prop_score (LogReg vs GBT)

`PropensityScorer`: джойн с тритом по `epk_id`, сравнение LogReg/GBT по ROC-AUC,
`prop_score` из лучшей модели. Диагностика overlap (доля в [0.1, 0.9]) важна для будущего PSM.

In [8]:
scorer = PropensityScorer(seed=SEED)
scorer.fit(emb, trt)                       # джойн по epk_id (в треате нет report_dt)
scores = scorer.transform(emb)

print("ROC-AUC по моделям:", {k: round(v, 4) for k, v in scorer.metrics_.items()})
print("выбранная модель:", scorer.best_model_name_)
_ov = scores.select(
    F.min(PROPENSITY_SCORE).alias("lo"), F.max(PROPENSITY_SCORE).alias("hi"),
    F.avg(((F.col(PROPENSITY_SCORE) >= 0.1) & (F.col(PROPENSITY_SCORE) <= 0.9)).cast("double")).alias("frac"),
).first()
print("overlap prop_score: min=%.3f max=%.3f доля в [0.1,0.9]=%.3f" % (_ov["lo"], _ov["hi"], _ov["frac"]))
scores.limit(5).toPandas()

ROC-AUC по моделям: {'logreg': 0.8779, 'gbt': 0.8571}
выбранная модель: logreg
overlap prop_score: min=0.000 max=1.000 доля в [0.1,0.9]=0.672


,epk_id,report_dt,prop_score
0,1000,2024-01-01,0.098797
1,1001,2024-03-01,0.894674
2,1002,2024-01-01,0.524035
3,1003,2024-02-01,0.111477
4,1004,2024-02-01,0.493333


## Выводы

- **Скрипт 1** отдаёт компактную таблицу `[epk_id, report_dt, red_0, …, red_{red_size-1}]` —
  понижение размерности сырых эмбеддингов (StandardScaler + PCA, in-time safe).
- **Скрипт 2** отдаёт `[epk_id, report_dt, prop_score]` от лучшего классификатора (LogReg/GBT по ROC-AUC).

**Что дальше:** `prop_score` → propensity-score matching по методологии Рубина
(построение пар контроль/тритмент по близости score, проверка баланса). Сам матчинг в этом
ноутбуке намеренно не реализован.

In [9]:
spark.stop()